# Example 7.4.1 / 1: Binary Times Bounded Continuous

Companion notebook for **7.4.1 Linearizing Products of Variables** in the AMPL lab Documentation, section "Binary Times Bounded Continuous". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Binary Times Bounded Continuous

For binary $b$, bounded $L \le x \le U$, and $z=bx$, use the exact convex-hull formulation (the
**convex hull** of a set is the smallest convex set containing it, so no tighter linear
description exists):

$$
Lb \le z \le Ub,\qquad
x-U(1-b) \le z \le x-L(1-b).
$$

To check the rows, fix `b` and `x`, then minimize and maximize `z`. If both give `b*x`, the rows
determine the product. The parameter `sense` switches the objective between the two directions,
with `L = -2` and `U = 5` so the interval crosses zero:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param L;                          # lower bound of x
param U >= L;                     # upper bound of x
param sense;                      # 1 to maximize z, -1 to minimize it
var b binary;                     # binary factor
var x >= L, <= U;                 # bounded continuous factor
var z;                            # stands for the product b*x

subject to ZLower0: z >= L * b;          # with b = 0: z >= 0
subject to ZUpper0: z <= U * b;          # with b = 0: z <= 0
subject to ZLower1: z >= x - U * (1-b);  # with b = 1: z >= x
subject to ZUpper1: z <= x - L * (1-b);  # with b = 1: z <= x
maximize Z: sense * z;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["L"] = -2
ampl.param["U"] = 5

In [5]:
# run.py
for b, x in [(0, 3), (1, -2), (1, 3)]:
    ampl.var["b"].fix(b)
    ampl.var["x"].fix(x)
    z = []
    for sense in (-1, 1):
        ampl.param["sense"] = sense
        ampl.solve(solver="highs", verbose=False)
        z.append(ampl.var["z"].value())
    print(f"b = {b}, x = {x}: z from {z[0]:g} to {z[1]:g}")

b = 0, x = 3: z from 0 to 0
b = 1, x = -2: z from -2 to -2
b = 1, x = 3: z from 3 to 3


### Pure AMPL: data.dat and model.run

`fix(value)` holds a variable at a value for the next solves. In each case the smallest and
largest `z` coincide at `b*x`.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param L := -2;
param U := 5;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
param zmin;
for {(bv, xv) in {(0, 3), (1, -2), (1, 3)}} {
    fix b := bv;
    fix x := xv;
    let sense := -1; solve; let zmin := z;
    let sense := 1; solve;
    printf "b = %g, x = %g: z from %g to %g\n", bv, xv, zmin, z;
}

b = 0, x = 3: z from 0 to 0
b = 1, x = -2: z from -2 to -2
b = 1, x = 3: z from 3 to 3


### Another way in amplpy

With a fractional `b`, the four rows leave a range of values, which is what a relaxation
explores. At `b = 0.5`, `x = 3` the product is 1.5, but `z` may lie anywhere in
`[x - U(1-b), U b] = [0.5, 2.5]`:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.param["L"] = -2
ampl.param["U"] = 5

In [11]:
# run.py
ampl.option["relax_integrality"] = 1
ampl.var["b"].fix(0.5)
ampl.var["x"].fix(3)
z = []
for sense in (-1, 1):
    ampl.param["sense"] = sense
    ampl.solve(solver="highs", verbose=False)
    z.append(ampl.var["z"].value())
print(f"b = 0.5, x = 3: z from {z[0]:g} to {z[1]:g}")

b = 0.5, x = 3: z from 0.5 to 2.5
